In [1]:
# ===========================================
# Libraries
# ===========================================
import numpy as np

In [2]:
# ===========================================
# Load chains
# ===========================================
# Certifique-se de que os caminhos estão corretos no seu ambiente local
path_base = "/home/brunowesley/projetos/MCMC-cosmo/Codes/GR-based/CPL/CC_SNe_BAO/flat_samples_CPL_cc_sne_bao.npy"
path_frb  = "/home/brunowesley/projetos/MCMC-cosmo/Codes/GR-based/CPL/CC_SNe_BAO_FRB/flat_samples_CPL_cc_sne_bao_frb.npy"

samples_base = np.load(path_base)
samples_frb  = np.load(path_frb)

# ===========================================
# Extract parameters (H0: index 0, Ob: index 2)
# ===========================================
H0_base, Ob_base = samples_base[:, 0], samples_base[:, 2]
H0_frb,  Ob_frb  = samples_frb[:, 0],  samples_frb[:, 2]

In [3]:
# ===========================================
# FoM & Stats Functions (Sincronizado com o Paper)
# ===========================================

def get_mcmc_stats(x, y):
    """
    Calculates FoM and statistics using percentile-based uncertainties (68% CL).
    This ensures consistency with GetDist tables and JCAP/PRD standards.
    """
    
    # 1. Calcular incertezas via percentis (68% CL)
    def get_sigma_eff(samples):
        low, med, high = np.percentile(samples, [16, 50, 84])
        # Incerteza simetrizada: média das barras de erro
        return ((high - med) + (med - low)) / 2.0

    sigma_x = get_sigma_eff(x)
    sigma_y = get_sigma_eff(y)

    # 2. Calcular Correlação (Pearson) das amostras
    # Usamos np.cov apenas para extrair o coeficiente de correlação rho
    data = np.vstack([x, y])
    cov_matrix = np.cov(data)
    rho = cov_matrix[0, 1] / (np.sqrt(cov_matrix[0, 0] * cov_matrix[1, 1]))
    
    # 3. Figure of Merit (Wang 2008)
    # FoM = 1 / (sigma_x * sigma_y * sqrt(1 - rho^2))
    # Note: Isso é equivalente a 1/sqrt(det(Cov)) mas usando os sigmas dos percentis
    fom = 1.0 / (sigma_x * sigma_y * np.sqrt(1 - rho**2))
    
    # 4. Skewness (Diagnóstico de Gaussianidade)
    def skewness(a):
        return np.mean((a - np.mean(a))**3) / (np.std(a)**3)
    
    return fom, sigma_x, sigma_y, rho, skewness(x), skewness(y)

# ===========================================
# Compute Results
# ===========================================

# Cálculos para Baseline e +FRB
fom_b, sH0_b, sOb_b, rho_b, skH0_b, skOb_b = get_mcmc_stats(H0_base, Ob_base)
fom_f, sH0_f, sOb_f, rho_f, skH0_f, skOb_f = get_mcmc_stats(H0_frb, Ob_frb)

improvement = (fom_f / fom_b - 1.0) * 100.0

# ===========================================
# Print Results
# ===========================================

print("\n" + "="*60)
print(f"{'CPL FoM Analysis (Percentile-based)':^60}")
print("="*60)

print("\n--- CC + SNe + BAO (Baseline) ---")
print(f"FoM (68% CL)  = {fom_b:.2f}")
print(f"σ(H0) eff     = {sH0_b:.3f}")
print(f"σ(Ωb) eff     = {sOb_b:.5f}")
print(f"ρ(H0,Ωb)      = {rho_b:.3f}")
print(f"Skew(H0)      = {skH0_b:.3f}")
print(f"Skew(Ωb)      = {skOb_b:.3f}")

print("\n--- CC + SNe + BAO + FRB ---")
print(f"FoM (68% CL)  = {fom_f:.2f}")
print(f"σ(H0) eff     = {sH0_f:.3f}")
print(f"σ(Ωb) eff     = {sOb_f:.5f}")
print(f"ρ(H0,Ωb)      = {rho_f:.3f}")
print(f"Skew(H0)      = {skH0_f:.3f}")
print(f"Skew(Ωb)      = {skOb_f:.3f}")

print("\n--- Comparison ---")
print(f"FoM increase  = {improvement:.2f}%")
print(f"Precision gain in Ωb = {(1 - sOb_f/sOb_b)*100:.2f}%")
print("="*60)


            CPL FoM Analysis (Percentile-based)             

--- CC + SNe + BAO (Baseline) ---
FoM (68% CL)  = 133.21
σ(H0) eff     = 1.563
σ(Ωb) eff     = 0.00504
ρ(H0,Ωb)      = 0.304
Skew(H0)      = 0.019
Skew(Ωb)      = 2.391

--- CC + SNe + BAO + FRB ---
FoM (68% CL)  = 259.30
σ(H0) eff     = 1.364
σ(Ωb) eff     = 0.00285
ρ(H0,Ωb)      = -0.134
Skew(H0)      = -0.055
Skew(Ωb)      = 0.230

--- Comparison ---
FoM increase  = 94.65%
Precision gain in Ωb = 43.42%
